# Animal classifier!

In [80]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn
from torchvision import transforms

In [81]:
ckpt = torch.load("animals_resnet18.pt", map_location="cpu")
classes = ckpt["classes"]

model = resnet18(weights=None)                       # no pretrained download needed
model.fc = nn.Linear(model.fc.in_features, len(ckpt["classes"]))
model.load_state_dict(ckpt["state_dict"])
model.eval()                                          # always, for inference


tfm = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(ckpt["norm_mean"], ckpt["norm_std"]),
    ]
)

def predict(img):
    with torch.no_grad():
        probs = model(tfm(img).unsqueeze(0)).softmax(dim=1)[0]
        return classes[probs.argmax()], probs.max().item()

In [82]:
# img = Image.open("animals/kangaroo/0024.jpg")#.convert("RGB")
# with torch.no_grad():
#     probs = model(tfm(img).unsqueeze(0)).softmax(dim=1)[0]
    
# print(classes[probs.argmax()], probs.max().item())

In [83]:
import ipywidgets as widgets
from PIL import Image
import matplotlib.pyplot as plt
import io

In [84]:
uploader = widgets.FileUpload(accept="image/*", multiple=False)
out_pl = widgets.Output()
lbl_pred = widgets.Label()

In [85]:
def on_click_classify(change):
    item = uploader.value[0]
    img = Image.open(io.BytesIO(item["content"]))

    out_pl.clear_output()  

    with out_pl:
        fig, ax = plt.subplots(figsize=[3,3])
        ax.imshow(img)
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    pred = predict(img)
    lbl_pred.value = f'Prediction: {pred[0]}; Probability: {pred[1]:.04f}'

In [86]:
uploader.observe(on_click_classify, names=['value'])

In [87]:
display(
    widgets.VBox(
        [
            widgets.Label('Select your animal!'),
            uploader,
            out_pl,
            lbl_pred
        ]
    )
)